# Analysis of Symbolic Regression equations
A manual analysis of selected equations.

In [1]:
import numpy as np
import re as regex
import openml
import sympy as sp

from scipy.optimize import minimize
from sklearn.metrics import r2_score

def get_task_clean_data_and_name(task_id) :
    """
    Get the data and the name of the dataset and target for a given task ID.
    """
    task = openml.tasks.get_task(task_id, download_splits=True,
                                     download_data=True, download_qualities=True,
                                     download_features_meta_data=True)
        
    # the 'task' object above contains a lot of useful information,
    # like the name of the target variable and the id of the dataset
    df_X, df_y = task.get_X_and_y('dataframe')
    
    # check if there is any missing value
    # here below there is a sum().sum() because it is adding up missing values
    # in rows AND THEN in columns
    missing_data = df_X.isnull().sum().sum() + df_y.isnull().sum()
    
    if missing_data > 0 :
        # we actually have to go with a task/dataset-specific correction, I think,
        # as there are only two datasets with missing values
        if task_id == 361268 : # dataset fps_benchmark
            # this task has several columns with A LOT of missing data,
            # so we are just going to drop them
            df_X.dropna(axis=1, inplace=True)
        elif task_id == 361616 : # dataset Moneyball
            # again, a few columns with 800/1200 missing values, get dropped
            df_X.dropna(axis=1, inplace=True)
    
    # check if there are any categorical columns
    df_categorical = df_X.select_dtypes(include=['category', 'object'])
    categorical_features = df_categorical.shape[1]
    
    # convert categorical columns to numerical values
    for c in df_categorical.columns :
        df_X[c] = df_X[c].astype('category') # double-check that it is treated as a categorical column
        df_X[c] = df_X[c].cat.codes # replace values with category codes (automatically computed)
    
    X = df_X.values
    y = df_y.values
    
    # let's also get the name of the dataset
    dataset = task.get_dataset()
        
    return df_X, df_y, dataset, task, missing_data, categorical_features

## abalone

In [ ]:
task_id = 361234
personalized_equation = "-0.0323031301921013 + (0.000204087575289272*x1 - 0.000185365221986661*x4 + 1.54799300186112e-6*x6 + 7.11224454083269e-5*x7)/(2.95765583621071e-5*x5 + 3.24193098237202e-6)"
personalized_equation = "3.73667019003373 + (10611898646893.3*x1 + 7007239420672.58*x4 - 16363792324998.6*x6 + 23095740989.1673*x7)/(1325768473061.23*x5 + 353966817487.809)"
personalized_equation = "-0.0323031301921013 + (-0.000185365221986661*x1 + 7.11224454083269e-5*x4 + 1.54799300186112e-6*x6 + 0.000204087575289272*x7)/(2.95765583621071e-5*x5 + 3.24193098237202e-6)"
personalized_equation = "6.0091949922108 + (-0.000396651914382866*x1 + 0.136164661078217*x4 - 0.0322536530441093*x6 + 0.0602221058849336*x7)/(0.00544078034214355 - 0.000993963605486663*x5)"
personalized_equation = "-x1 + x4 - (x1 - 0.513) + (x3 + (x4 + x7 - (x5 + x6))/(x5+0.143)) * 5.90 + 3.01"


# parse equation, convert it first to symbolic representation, then to a lambdified executable expression
expr = sp.sympify(personalized_equation)
print("Personalized equation:", expr)
variables = sorted([s for s in expr.free_symbols if s.name.startswith('x')], key=lambda s: s.name)
print("Variables of the personalized equation:", variables)
features = [str(v) for v in variables]
lambdified_function = sp.lambdify(variables, expr, modules='numpy')

# load dataset
df_X, df_y, dataset, task, missing_data, categorical_features = get_task_clean_data_and_name(task_id)
df_X.columns = ["x%d" % i for i in range(1, len(df_X.columns)+1)]
print(df_X.columns)
X = df_X[features].values
y = df_y.values

y_pred = lambdified_function(*X.T)
r2_value = r2_score(y, y_pred)
print("R2 value of the lambdified expression: %.4f" % r2_value)

Personalized equation: -2*x1 + 5.9*x3 + x4 + 3.523 + 5.9*(x4 - x5 - x6 + x7)/(x5 + 0.143)
Variables of the personalized equation: [x1, x3, x4, x5, x6, x7]
Index(['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'], dtype='object')
R2 value of the lambdified expression: -11.4366


## Moneyball

In [ ]:
task_id = 361616
personalized_equation = ""

# parse equation, convert it first to symbolic representation, then to a lambdified executable expression
expr = sp.sympify(personalized_equation)
print("Personalized equation:", expr)
variables = sorted([s for s in expr.free_symbols if s.name.startswith('x')], key=lambda s: s.name)
print("Variables of the personalized equation:", variables)
lambdified_function = sp.lambdify(variables, expr, modules='numpy')

# load dataset
df_X, df_y, dataset, task, missing_data, categorical_features = get_task_clean_data_and_name(task_id)
df_X.columns = ["x%d" % i for i in range(1, len(df_X.columns)+1)]
print(df_X.columns)
X = df_X[[str(v) for v in variables]].values
y = df_y.values

y_pred = lambdified_function(*X.T)
r2_value = r2_score(y, y_pred)
print("R2 value of the lambdified expression: %.4f" % r2_value)

Personalized equation: 0.617*x4 + x5*(-0.002*x3 + 27.636*x6) - 165.422
Variables of the personalized equation: [x3, x4, x5, x6]
Index(['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10'], dtype='object')
R2 value of the lambdified expression: 0.8009


## 361258

In [3]:
from sympy import cse, factor, simplify, latex
from sympy.parsing.latex import parse_latex
from sympy import Symbol

In [ ]:
latex_equation = r'\left(x_{2} \left(-0.177\right) + \left(\left(\left(0.623 - \left(x_{2} + x_{5}\right) 0.396\right) 0.320 + \cos{\left(x_{1} \left(-1.01\right) + \left(x_{0} + 0.230\right) x_{2} \left(-0.240\right) - 0.478 \right)}\right) x_{0} \left(-0.223\right) + \left(- x_{1} + x_{4} + 0.625\right) 0.0946 + \cos{\left(x_{2} - 0.0909 \right)} \cos{\left(- (x_{7} - \left(x_{3} + x_{5} - x_{6} \left(-0.415\right) + x_{7} - \left(x_{6} - x_{7} + 0.508\right) \left(-0.573\right) + 0.758\right)) - 0.691 \right)}\right) 0.308 + \sin{\left(x_{5} - \left(x_{0} \left(-0.117\right) + x_{6} \left(-0.914\right)\right) + 0.415 \right)} \left(-0.0757\right) + 0.782\right) 0.873 + \left(\cos{\left(- x_{4} - x_{4} + x_{4} \cdot 0.440 + 0.136 \right)} x_{3} \left(-0.409\right) + x_{4} + \left(x_{2} + x_{7}\right) \frac{1}{0.617} \cdot 0.205 \left(x_{4} \left(- x_{1} + x_{6}\right) \left(-0.427\right) + 0.906\right) + \left(- x_{7} + x_{7} + x_{7} - \left(x_{7} + \left(x_{4} \cdot 2.03 + 0.979\right) \left(x_{2} \left(-0.516\right) + \left(- x_{1} + x_{2}\right) \left(-0.306\right) + 0.584\right) + \left(x_{1} + \cos{\left(x_{3} \right)} + 1.76\right) \left(-0.341\right) - 1.23 - -0.0957\right)\right) \left(-0.596\right) + \cos{\left(x_{4} \right)}\right) \frac{1}{0.575} \cos{\left(- x_{5} + \left(x_{6} + x_{6} - x_{7}\right) \left(-0.547\right) \right)} 0.0569'
expr = parse_latex(latex_equation)
print("Original expression:", expr)

Original expression: (cos((-x_{5} + (-x_{7} + (x_{6} + x_{6}))*(-0.547))*0.0569)/0.575)*(((((x_{2} + x_{7})/0.617)*(0.205*(0.906 + x_{4}(-x_{1} + x_{6})*(-0.427))) + (x_{4} + cos((x_{3}*(-0.409))*((x_{4}*0.44 + (-x_{4} - x_{4})) + 0.136)))) + ((x_{7} + (-x_{7} + x_{7})) - ((((x_{7} + (x_{4}*2.03 + 0.979)*(((-x_{1} + x_{2})*(-0.306) + x_{2}(-0.516)) + 0.584)) + ((x_{1} + cos(x_{3})) + 1.76)*(-0.341)) - 1.23) + 0.0957))*(-0.596)) + cos(x_{4})) + (((((((0.623 - 0.396*(x_{2} + x_{5}))*0.32 + cos(((x_{0} + 0.23)*x_{2}(-0.24) + x_{1}(-1.01)) - 0.478))*x_{0}(-0.223) + ((-x_{1} + x_{4}) + 0.625)*0.0946) + cos(x_{2} - 0.0909)*cos((x_{3} + x_{5} - 1*((x_{6} - x_{7}) + 0.508)*(-0.573) - x_{6}(-0.415) + 0.758) - 0.691))*0.308 + x_{2}(-0.177)) + sin(((x_{5} - (x_{0}(-0.117) + x_{6}(-0.914))) + 0.415)*(-0.0757))) + 0.782)*0.873


In [6]:
def replace_constants(expr):
    constants = sorted(
        [x for x in expr.atoms() if x.is_number],
        key=str
    )

    mapping = {
        c: Symbol(f"c{i}")
        for i, c in enumerate(constants)
    }

    return expr.xreplace(mapping), mapping

def rename_x_variables(expr):
    mapping = {}

    for s in expr.free_symbols:
        name = str(s)

        if name.startswith("x_{") and name.endswith("}"):
            k = name[3:-1]
            mapping[s] = Symbol(f"x{k}")

    return expr.xreplace(mapping)

In [ ]:
expr, mapping = replace_constants(expr)
expr = rename_x_variables(expr)
print("Expression with constants replaced by symbols:", expr)

Expression with constants replaced by symbols: c33**c18*(c15*(c0*x7 + c18*x7 + c18*(c20 + c23 + c7*(c43 + x1 + cos(x3)) + x7 + (c42 + c44*x4)*(c34 + c6*(c18*x1 + x2) + x_{2}(c12)))) + c25*c35**c18*(x2 + x7)*(c10*x_{4}(c18*x1 + x6) + c41) + x4 + cos(x4) + cos(c8*x3*(c0*c18*x4 + c24 + c31*x4)))*cos(c21*(c13*(c0*x6 + c18*x7) + c18*x5)) + c40*(c27*(c22*(c18*x1 + c37 + x4) + (c28*(c18*c29*(x2 + x5) + c36) + cos(c11 + (c26 + x0)*x_{2}(c5) + x_{1}(c19)))*x_{0}(c4) + cos(c1 + x2)*cos(c14*c18*(c18*x7 + c32 + x6) + c16 + c18*x_{6}(c9) + c38 + x3 + x5)) + c39 + x_{2}(c3) + sin(c0*(c18*(x_{0}(c2) + x_{6}(c17)) + c30 + x5)))


In [ ]:
from sympy import cse, factor, cancel

expr = parse_latex(latex_equation)
expr = expr.doit()
print(expr)
print("- canceling...")
expr = cancel(expr)
print(expr)
print("- factoring...")
expr = factor(expr)
print(expr)
print("- collecting common sub-expressions...")
subs, expr = cse(expr)

print(expr)
expr, mapping = replace_constants(expr[0])
print(expr)
expr = rename_x_variables(expr)
print(expr)

- canceling...
- factoring...
- collecting common sub-expressions...
[0.268884*x0*cos(x9*x_{0} + 0.23*x9 + x_{1}(-1.01) - 0.478) + 0.05360471424*x0 - 2.96794768695652*x1*x_{2} - 2.96794768695652*x1*x_{5} - 2.96794768695652*x10*x3 - 2.96794768695652*x10*x5 + 1.0147547826087*x2*x6 - 0.0429389495652174*x2*x_{1} - 0.353453913043478*x2*cos(x_{3}) - 1.2051887026087*x2 + 0.212999940334015*x3 + 2.10413913043478*x4*x6 + 2.96794768695652*x4 + 0.523514903812275*x5 + 2.96794768695652*x7*cos(x_{4}) + 2.96794768695652*x7*cos(0.63804*x_{3}*x_{4} - 0.055624*x_{3}) + 2.96794768695652*x8*x_{1} - 2.96794768695652*x8*x_{2} - 0.0254364264*x_{1} + 0.0254364264*x_{4} + 0.873*x_{2}(-0.177) - 0.873*sin(0.0757*x_{5} - 0.0757*x_{0}(-0.117) - 0.0757*x_{6}(-0.914) + 0.0314155) + 0.268884*cos(x_{2} - 0.0909)*cos(x_{3} + x_{5} + 0.573*x_{6} - 0.573*x_{7} - x_{6}(-0.415) + 0.358084) + 0.6985837665]
c0*x_{1} + c1*x2*x_{1} + c11*sin(c18 + c20*x_{5} + c3*x_{0}(c5) + c3*x_{6}(c12)) + c15*x2 + c16*x1*x_{2} + c16*x1*x_{5} 